# Week 11 - 2026-06-19 실습

## 📅 오늘 학습 주제: Theme 46: LangGraph 기반 StateGraph 에이전트 설계 및 제어 실습
- **학습 목표**:
  - LangGraph의 핵심 구성 요소인 State(상태), Nodes(노드), Edges(엣지)의 아키텍처적 관계를 명확히 이해하고 구현합니다.
  - `StateGraph`를 활용하여 복잡하고 순환 가능한(Cyclic) 에이전트 추론 루프 및 워크플로우를 설계합니다.
  - LLM의 추론 결과에 따라 흐름을 분기하는 `Conditional Edge`를 정의하여 지능적인 제어 흐름을 완성합니다.
  - 에이전트의 예외 및 오동작 방지를 위한 `recursion_limit` 및 State 병합 규칙(Annotated/reducer)을 실습합니다.
- **Senior Mentor의 핵심 가이드**:
  - 단순한 선형 체인(LCEL)을 넘어 루프가 발생할 수 있는 복잡한 에이전트 시스템은 흐름 제어 프레임워크(LangGraph)가 필수적입니다.
  - 상태 기반 아키텍처에서는 전역 상태(State)의 설계가 시스템 전체의 안정성을 결정하므로, 상태 변경 시의 갱신 규칙(Annotated, operator.add 등)과 불변성 유지를 면밀히 파악해야 합니다.

In [1]:
# 1. 프로젝트 경로 추가 및 환경 설정 로드
import sys
from pathlib import Path

# 현재 작업 디렉토리의 상위(프로젝트 루트)를 Python path에 추가
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from config import CONTENT_DIR, GOOGLE_AI_API_KEY
print(f"[상태] 프로젝트 루트 경로: {project_root}")
print(f"[상태] Gemini API 키 로드 여부: {'성공' if GOOGLE_AI_API_KEY else '실패'}")

[상태] 프로젝트 루트 경로: /home/hong/project/ai-camp-note
[상태] Gemini API 키 로드 여부: 성공


In [2]:
# 2. 임베딩 모델 및 LLM 구성
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.chat_models import init_chat_model
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

CHAT_MODEL = 'google_genai:gemma-4-26b-a4b-it'
llm = init_chat_model(CHAT_MODEL, api_key=GOOGLE_AI_API_KEY, temperature=0.1)
print(f"[준비] 256차원 임베딩 및 {CHAT_MODEL} LLM이 준비되었습니다.")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[준비] 256차원 임베딩 및 google_genai:gemma-4-26b-a4b-it LLM이 준비되었습니다.


## 🛠️ 실습 1: LangGraph 기본 StateGraph 및 Node, Edge 구축

LangGraph에서 가장 중요한 개념인 **상태(State)**를 정의하고, 각 상태를 거치며 변수를 조작하는 **노드(Node)**와 노드 간의 흐름을 잇는 **엣지(Edge)**를 정의하여 가장 기본적인 비순환 그래프(DAG)를 빌드해봅니다.

In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# 1. State 정의 (노드 간에 전달되고 갱신될 전역 상태 데이터)
class SimpleState(TypedDict):
    question: str
    messages: list[str]
    current_node: str

# 2. Node 정의 (이전 상태를 입력받아 변경될 상태 값을 딕셔너리로 반환)
def start_node(state: SimpleState) -> dict:
    print("\n--- Node 1 (Start) ---")
    current_messages = state.get("messages", [])
    new_messages = current_messages + ["Start Node를 방문했습니다."]
    return {"messages": new_messages, "current_node": "start_node"}

def process_node(state: SimpleState) -> dict:
    print("\n--- Node 2 (Process) ---")
    current_messages = state.get("messages", [])
    new_messages = current_messages + ["Process Node에서 질문을 분석했습니다."]
    return {"messages": new_messages, "current_node": "process_node"}

def end_node(state: SimpleState) -> dict:
    print("\n--- Node 3 (End) ---")
    current_messages = state.get("messages", [])
    new_messages = current_messages + ["End Node를 방문했습니다. 흐름을 종료합니다."]
    return {"messages": new_messages, "current_node": "end_node"}

# 3. StateGraph 구축
workflow = StateGraph(SimpleState)

# 노드 등록
workflow.add_node("start", start_node)
workflow.add_node("process", process_node)
workflow.add_node("end", end_node)

# 엣지 연결 (START -> start -> process -> end -> END)
workflow.add_edge(START, "start")
workflow.add_edge("start", "process")
workflow.add_edge("process", "end")
workflow.add_edge("end", END)

# 컴파일
app = workflow.compile()
print("[상태] Simple StateGraph 컴파일 완료")

[상태] Simple StateGraph 컴파일 완료


In [4]:
# 4. 실행 테스트
initial_state = {"question": "오늘 날씨 어때?", "messages": [], "current_node": ""}
result = app.invoke(initial_state)

print("\n================ [최종 상태 결과] ================")
print(f"질문: {result['question']}")
print(f"마지막 노드: {result['current_node']}")
print("메시지 기록:")
for msg in result["messages"]:
    print(f" - {msg}")


--- Node 1 (Start) ---

--- Node 2 (Process) ---

--- Node 3 (End) ---

================ [최종 상태 결과] ================
질문: 오늘 날씨 어때?
마지막 노드: end_node
메시지 기록:
 - Start Node를 방문했습니다.
 - Process Node에서 질문을 분석했습니다.
 - End Node를 방문했습니다. 흐름을 종료합니다.


## 🤖 실습 2: Conditional Edge 기반의 에이전트 분기 제어 및 RAG/도구 결합

실무 에이전트 아키텍처에서는 질문을 분류하여 외부 지식(RAG/DB)을 찾아와야 할지, 아니면 LLM 자체 지식으로 바로 응답할지 스스로 선택해야 합니다.
이를 위해 질문의 성격을 분류하고, 결과에 따라 다이나믹하게 노드 이동 경로를 분기시키는 `Conditional Edge` 및 라우팅 함수를 연동해봅니다.

In [5]:
from typing import Literal
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool

# 1. 외부 사내 규정 조회 도구 정의
@tool
def search_local_db(query: str) -> str:
    """사내 규정 및 로컬 DB 문서를 검색합니다."""
    # 간단한 Mocking 데이터 반환
    if "경비" in query or "출장" in query or "서식" in query:
        return "SK-PET-2026 규정: 법인카드 경비 신청서(서식 4호)는 결제일 3영업일 이내에 온라인 그룹웨어를 통해 제출해야 한다."
    return "요청하신 주제에 대한 관련 사내 규정 문서를 찾지 못했습니다."

# 2. AgentState 정의
class AgentState(TypedDict):
    question: str
    search_needed: bool
    context: str
    answer: str

# 3. 각 노드 함수 정의

# [노드 1] 질문이 회사 내부 정보(경비 규정 등) 조회를 요구하는지 분석
def classify_question_node(state: AgentState) -> dict:
    print("\n--- [노드] Classify Question ---")
    question = state["question"]
    
    # LLM 구조화된 출력을 활용하여 판단
    from pydantic import BaseModel, Field
    
    class RouteDecision(BaseModel):
        search_needed: bool = Field(
            description="질문에 답하기 위해 사내 규정 문서나 DB 검색이 필요하면 True, 일반 상식이거나 대화라면 False"
        )
    
    structured_llm = llm.with_structured_output(RouteDecision)
    prompt = f"다음 질문에 답하기 위해 사내 규정이나 문서 검색이 필요합니까?\n질문: {question}"
    
    decision = structured_llm.invoke([
        SystemMessage(content="너는 사용자의 질문이 사내 규정이나 문서 DB 검색을 필요로 하는지 객관적으로 분류하는 전문가이다."),
        HumanMessage(content=prompt)
    ])
    
    print(f"[분석 결과] DB 검색 필요 여부: {decision.search_needed}")
    return {"search_needed": decision.search_needed}

# [노드 2] DB 및 사내 규정 문서 검색을 수행
def search_db_node(state: AgentState) -> dict:
    print("\n--- [노드] Search Local DB ---")
    question = state["question"]
    
    # 정의한 search_local_db 도구 실행
    db_result = search_local_db.invoke(question)
    print(f"[조회 결과] 검색된 문맥: {db_result}")
    return {"context": db_result}

# [노드 3] 획득한 문맥(Context)을 조합하여 최종 답변을 생성
def generate_answer_node(state: AgentState) -> dict:
    print("\n--- [노드] Generate Answer ---")
    question = state["question"]
    context = state.get("context", "")
    
    if context:
        prompt = f"다음 사내 문서(Context)를 철저히 참고하여 질문에 사실만으로 답하세요.\n\n[Context]\n{context}\n\n[Question]\n{question}"
    else:
        prompt = f"다음 질문에 친절하고 상세하게 답하세요. (사내 문서 없음)\n\n[Question]\n{question}"
        
    response = llm.invoke(prompt)
    return {"answer": response.content}

# 4. 라우팅 조건 함수 (분석된 상태 값에 따라 이동할 다음 노드 이름을 리턴)
def router_decision(state: AgentState) -> Literal["search", "generate"]:
    if state["search_needed"]:
        return "search"
    return "generate"

# 5. StateGraph 설정 및 조립
agent_workflow = StateGraph(AgentState)

# 노드들 추가
agent_workflow.add_node("classify", classify_question_node)
agent_workflow.add_node("search_db", search_db_node)
agent_workflow.add_node("generate_answer", generate_answer_node)

# START -> classify
agent_workflow.add_edge(START, "classify")

# classify 노드에서 라우팅 조건에 따라 분기 처리
agent_workflow.add_conditional_edges(
    "classify",
    router_decision,
    {
        "search": "search_db",
        "generate": "generate_answer"
    }
)

# search_db -> generate_answer
agent_workflow.add_edge("search_db", "generate_answer")

# generate_answer -> END
agent_workflow.add_edge("generate_answer", END)

# 컴파일
agent_app = agent_workflow.compile()
print("[상태] Agent StateGraph 컴파일 및 조립 성공")

[상태] Agent StateGraph 컴파일 및 조립 성공


In [6]:
# 테스트 1: 사내 정보 조회가 필요한 질문 (search_db를 통과해야 함)
test_state_1 = {
    "question": "출장 후에 법인카드 결제 영수증은 언제까지 올려야 해요?",
    "search_needed": False,
    "context": "",
    "answer": ""
}
result_1 = agent_app.invoke(test_state_1)

print("\n================ [테스트 1 최종 결과] ================")
print(f"질문: {result_1['question']}")
print(f"참고문맥: {result_1['context']}")
print(f"최종답변: {result_1['answer']}")


--- [노드] Classify Question ---
[분석 결과] DB 검색 필요 여부: True

--- [노드] Search Local DB ---
[조회 결과] 검색된 문맥: SK-PET-2026 규정: 법인카드 경비 신청서(서식 4호)는 결제일 3영업일 이내에 온라인 그룹웨어를 통해 제출해야 한다.

--- [노드] Generate Answer ---

================ [테스트 1 최종 결과] ================
질문: 출장 후에 법인카드 결제 영수증은 언제까지 올려야 해요?
참고문맥: SK-PET-2026 규정: 법인카드 경비 신청서(서식 4호)는 결제일 3영업일 이내에 온라인 그룹웨어를 통해 제출해야 한다.
최종답변: [{'type': 'thinking', 'thinking': '*   Context: "SK-PET-2026 규정: 법인카드 경비 신청서(서식 4호)는 결제일 3영업일 이내에 온라인 그룹웨어를 통해 제출해야 한다." (SK-PET-2026 Regulation: Corporate card expense application forms (Form 4) must be submitted via the online groupware within 3 business days of the payment date.)\n    *   Question: "출장 후에 법인카드 결제 영수증은 언제까지 올려야 해요?" (When should I upload the corporate card payment receipt after a business trip?)\n\n    *   The context specifies the deadline for the "법인카드 경비 신청서(서식 4호)" (Corporate card expense application form).\n    *   The deadline is "결제일 3영업일 이내" (within 3 business days of the payment date).\n    *

In [7]:
# 테스트 2: 일반 대화 및 상식형 질문 (DB 검색을 스킵하고 바로 답변을 얻어야 함)
test_state_2 = {
    "question": "인공지능 에이전트와 LLM의 차이점을 한 단어로 표현해줘.",
    "search_needed": False,
    "context": "",
    "answer": ""
}
result_2 = agent_app.invoke(test_state_2)

print("\n================ [테스트 2 최종 결과] ================")
print(f"질문: {result_2['question']}")
print(f"참고문맥: {result_2['context']} (비어있어야 정상)")
print(f"최종답변: {result_2['answer']}")


--- [노드] Classify Question ---
[분석 결과] DB 검색 필요 여부: False

--- [노드] Generate Answer ---

================ [테스트 2 최종 결과] ================
질문: 인공지능 에이전트와 LLM의 차이점을 한 단어로 표현해줘.
참고문맥:  (비어있어야 정상)
최종답변: [{'type': 'thinking', 'thinking': '\n*   Question: "What is the difference between an AI Agent and an LLM? Express it in one word."\n*   Constraint: Be kind and detailed.\n*   Context: No internal company documents provided.\n\n    *   *LLM (Large Language Model):* A model trained on vast amounts of text to predict the next token. It\'s a "brain" or a "knowledge base." It processes input and generates output. It\'s passive (waits for a prompt).\n    *   *AI Agent:* A system that uses an LLM as its core reasoning engine but can *act* on the world. It has goals, can use tools (APIs, web search), can plan steps, and can iterate based on feedback. It\'s active.\n\n    *   *Brain vs. Body?* (Too simple)\n    *   *Knowledge vs. Action?* (Two words)\n    *   *Thinking vs. Doing?* (Two words)\n    

## 🛠️ 실습 3: Recursion Limit 제어 및 루프 방어

LangGraph 에이전트는 복잡한 피드백 루프를 처리하는 데 특화되어 있으나, 흐름이 순환 구조를 가질 경우 논리 결함으로 인해 무한 루프(Infinite Loop)에 빠질 위험이 존재합니다.
이를 차단하고 안전하게 런타임을 보호하기 위한 `recursion_limit` 실행 옵션을 알아보고 직접 에러를 포착하여 대응하는 실습을 진행합니다.

In [8]:
# 일부러 무한 루프를 도는 그래프를 조립하여 방어선을 테스트합니다.
class LoopState(TypedDict):
    counter: int

def node_a(state: LoopState) -> dict:
    val = state.get("counter", 0) + 1
    print(f"[Node A 실행] Counter: {val}")
    return {"counter": val}

def node_b(state: LoopState) -> dict:
    val = state.get("counter", 0) + 1
    print(f"[Node B 실행] Counter: {val}")
    return {"counter": val}

loop_workflow = StateGraph(LoopState)
loop_workflow.add_node("A", node_a)
loop_workflow.add_node("B", node_b)

loop_workflow.add_edge(START, "A")
loop_workflow.add_edge("A", "B")
loop_workflow.add_edge("B", "A")  # 순환 구조(무한 루프 유도)

loop_app = loop_workflow.compile()
print("[상태] 순환 무한 루프 StateGraph 빌드 완료")

[상태] 순환 무한 루프 StateGraph 빌드 완료


In [ ]:
# 5단계 이상의 흐름 차단이 걸리는지 5로 설정하고 실행
try:
    print("--- 루프 실행 시작 ---")
    # config에 recursion_limit 설정 전달 (기본값은 25)
    loop_app.invoke({"counter": 0}, config={"recursion_limit": 5})
except Exception as e:
    print(f"\n[예외 포착] 무한 루프 임계치 초과 에러가 정상 작동했습니다:\n{e}")

--- 루프 실행 시작 ---
[Node A 실행] Counter: 1
[Node B 실행] Counter: 2
[Node A 실행] Counter: 3
[Node B 실행] Counter: 4
[Node A 실행] Counter: 5

[예외 포착] 무한 루프 임계치 초과 에러가 정상 작동했습니다:
Recursion limit of 5 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT


## 📝 오늘의 실습 정리 및 심화 학습 과제

1. **학습 성과 요약**
   - LangGraph의 구조(State, Node, Edge, Conditional Edge)를 이해하고, LLM을 활용한 스마트 라우터 분기 워크플로우를 완수했습니다.
   - `recursion_limit`를 이용해 에이전트 시스템에 발생할 수 있는 무한 루프 폭주를 강제 제어하는 현업 방어선 기법을 학습했습니다.

2. **심화 학습 과제 (스스로 시도해보기)**
   - **과제 1**: `search_db_node`에서 검색 결과를 찾지 못했을 때(Mock DB에 매칭되지 않을 때) 에러 상태 코드를 반환하거나 일반적인 답변으로 유도하는 **예외 가드레일 조건식**을 추가해보세요.
   - **과제 2**: `StateGraph`의 State를 확장하여 이전 대화 기록(`messages: list`)을 누적하여 저장하고, 최종 노드에서 과거 대화 맥락까지 함께 인지하여 답변하도록 상태와 프롬프트를 개선해보세요.

In [3]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

MEETING_DIR = CONTENT_DIR / "meeting_notes"

def load_meeting_docs(data_dir: Path):
    all_docs = []
    for pattern in ["**/*.txt", "**/*.md"]:
        loader = DirectoryLoader(
            str(data_dir),
            glob=pattern,
            loader_cls=TextLoader,
            loader_kwargs={"encoding": "utf-8"},
        )
        all_docs.extend(loader.load())
    return all_docs

meeting_docs = load_meeting_docs(MEETING_DIR)
print(f"로드된 회의록: {len(meeting_docs)}개")

/tmp/ipykernel_91298/1792344603.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


로드된 회의록: 7개


In [4]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from kiwipiepy import Kiwi
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

def chunk_meeting_note(file_content: str, file_path: str):
    # 1. 헤더 기준 분할 정의
    headers_to_split_on = [
        ("##", "Header 1"),
        ("###", "Header 2"),
    ]
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on, 
        strip_headers=False
    )
    
    # 2. 텍스트 분할
    chunks = markdown_splitter.split_text(file_content)
    
    # 3. 회의 메타데이터 추출 (간단한 파싱 예시)
    # 실제로는 정규식 등을 활용해 '일시', '장소', '참석자'를 객체화합니다.
    meeting_info = ""
    lines = file_content.split("\n")
    for line in lines:
        if line.startswith("- 일시:") or line.startswith("- 장소:") or line.startswith("- 참석자:"):
            meeting_info += line + "\n"
        if line.startswith("## 1."): # 메타데이터 영역이 끝나면 중단
            break
            
    # 4. 각 청크에 공통 메타데이터 주입 및 텍스트 보강
    refined_chunks = []
    for chunk in chunks:
        # 회의 정보가 누락된 청크에 정보 보강
        if "회의 정보" not in chunk.metadata.get("Header 1", ""):
            # RAG 성능 향상을 위해 텍스트 상단에 회의 컨텍스트를 프레임으로 씌워줌
            chunk.page_content = f"[회의 컨텍스트]\n{meeting_info.strip()}\n\n[세부 내용]\n{chunk.page_content}"
            
        # 메타데이터 사전(dict)에도 파일 경로 및 공통 정보 추가
        chunk.metadata["source"] = file_path
        refined_chunks.append(chunk)
        
    return refined_chunks

# 1. 로드된 문서(meeting_docs)를 순회하며 청킹을 수행합니다.
all_chunks = []

for doc in meeting_docs:
    # doc.page_content: 문서의 텍스트 내용
    # doc.metadata["source"]: 파일의 절대 경로
    file_content = doc.page_content
    file_path = doc.metadata.get("source", "unknown")
    
    # 정의된 청킹 함수 호출
    chunks = chunk_meeting_note(file_content, file_path)
    all_chunks.extend(chunks)

# 2. 결과 확인
print(f"전체 회의록 문서 개수: {len(meeting_docs)}개")
print(f"청킹을 통해 생성된 총 청크 수: {len(all_chunks)}개")

# 첫 번째 청크 예시 확인
if all_chunks:
    print("\n--- 첫 번째 청크 내용 샘플 ---")
    print(all_chunks[0].page_content)
    print("메타데이터:", all_chunks[0].metadata)

from langchain_chroma import Chroma

print("[상태] 인메모리 Chroma 벡터 저장소를 생성 중입니다...")

# persist_directory를 지정하지 않으면 인메모리 모드로 작동합니다.
vectorstore = Chroma.from_documents(
    documents=all_chunks,
    embedding=embeddings
)

# 1. BM25 Sparse Retriever 생성 (형태소 분석이나 토큰 기반 키워드 검색)
# 1. Kiwi 형태소 분석기 초기화
kiwi = Kiwi()
# 2. 커스텀 토큰화 함수 정의 (문장에서 조사/어미를 제외하고 명사/동사/형용사 위주로 토큰 추출)
def kiwi_tokenize(text: str) -> list[str]:
    # 형태소 분석 수행
    tokens = kiwi.tokenize(text)
    
    # 3. 실무 팁: 의미를 가지는 실질 형태소 품사만 필터링하여 노이즈 차단
    # N(명사), V(동사/형용사), SN(숫자), SL(외국어) 등
    allowed_tags = ('NNG', 'NNP', 'NNB', 'NR', 'NP', 'VV', 'VA', 'SN', 'SL')
    
    return [
        token.form for token in tokens 
        if token.tag.startswith(allowed_tags)
    ]
bm25_retriever = BM25Retriever.from_documents(all_chunks, preprocess_func=kiwi_tokenize)
bm25_retriever.k = 2  # BM25에서 가져올 문서 개수

# 2. Chroma Dense Vector Retriever 생성
vector_retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2}
)

# 3. Ensemble Retriever로 두 리트리버 결합 (가중치 5:5 설정)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5]
)

cross_encoder = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")
reranker = CrossEncoderReranker(model=cross_encoder, top_n=3)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=reranker,
    base_retriever=ensemble_retriever,
)

print("[성공] BM25와 Chroma를 결합한 하이브리드 Ensemble Retriever가 구성되었습니다.")


전체 회의록 문서 개수: 7개
청킹을 통해 생성된 총 청크 수: 103개

--- 첫 번째 청크 내용 샘플 ---
# 프로젝트 킥오프 회의록  
## 회의 정보
- 일시: 2024년 1월 15일 (월) 14:00 ~ 15:30
- 장소: 대회의실 A
- 참석자: 김철수(PM), 이영희(백엔드 개발), 박민수(프론트엔드 개발), 정지원(디자인), 최동현(QA)  
---
메타데이터: {'Header 1': '회의 정보', 'source': '/home/hong/project/ai-camp-note/content/meeting_notes/meeting_2024_01.txt'}
[상태] 인메모리 Chroma 벡터 저장소를 생성 중입니다...


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

[성공] BM25와 Chroma를 결합한 하이브리드 Ensemble Retriever가 구성되었습니다.


In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

test_questions = [
    "1월 킥오프 회의에서 결정된 최종 릴리즈 일정은 언제인가요?",            # meeting_2024_01
    "스프린트 1에서 완료되지 못해 스프린트 2로 이월된 작업은 무엇인가요?",   # meeting_2024_02
    "PR #287 사용자 인증 모듈 리팩토링에 대한 리뷰 결정은 무엇이었나요?",    # meeting_2024_03
    "ABC 테크놀로지 고객사의 월간 활성 사용자(MAU) 수치는 어떻게 되나요?",  # meeting_2024_04
    "5월 14일 발생한 장애의 지속 시간과 영향 받은 사용자 수는?",          # meeting_2024_05
    "2024년 1분기 매출 실적과 목표 대비 달성률은 얼마인가요?",            # meeting_2024_06
    "SmartWork Pro 4.0 고객 요청 기능 1순위는 무엇인가요?",              # meeting_2024_07
]

# 1. RAG 시스템을 위한 프롬프트 템플릿 설계
# (할루시네이션 방지를 위해 회의록 정보만을 이용하도록 강력히 제약)
prompt = ChatPromptTemplate.from_template(
    "당신은 사내 회의록 내용을 기반으로 답변하는 전문 AI 비서입니다.\n"
    "아래 제공되는 회의록 정보([Context])만을 철저히 참고하여 질문에 사실 정보만으로 명확하게 답변해 주세요.\n"
    "회의록 내용에 답변의 근거가 없거나 모호한 경우, 억지로 지어내지 말고 '제공된 회의록에서 관련 정보를 찾을 수 없습니다.'라고 답변해 주세요.\n\n"
    "[Context]\n{context}\n\n"
    "[Question]\n{question}"
)

# 2. 검색된 문서들을 하나의 문자열로 결합하는 헬퍼 함수
def format_docs(docs):
    formatted = []
    for doc in docs:
        source_name = doc.metadata.get("source", "").split("/")[-1] # 파일명만 추출
        formatted.append(f"--- 파일 출처: {source_name} ---\n{doc.page_content}")
    return "\n\n".join(formatted)

rag_chain = (
    {
        "context": compression_retriever | format_docs, 
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# 4. 질문 리스트를 순회하며 테스트 실행
print("회의록 RAG 질의응답 테스트를 시작합니다.")
print("=" * 60)

for i, q in enumerate(test_questions, 1):
    print(f"\n[Q{i}] 질문: {q}")
    
    # RAG 검색 및 생성 실행
    response = rag_chain.invoke(q)
    
    # 가 Gemma 모델의 경우 가끔 내부 생각 구조(Thinking block)가 들어오거나 포맷이 정해져 있을 수 있으므로 그대로 출력합니다.
    print(f"[A{i}] 답변:")
    print(response)
    print("-" * 60)


회의록 RAG 질의응답 테스트를 시작합니다.

[Q1] 질문: 1월 킥오프 회의에서 결정된 최종 릴리즈 일정은 언제인가요?
[A1] 답변:
최종 릴리즈 일정은 2024년 4월 30일입니다.
------------------------------------------------------------

[Q2] 질문: 스프린트 1에서 완료되지 못해 스프린트 2로 이월된 작업은 무엇인가요?
[A2] 답변:
스프린트 1에서 완료되지 못해 스프린트 2로 이월된 작업은 API 인증 모듈입니다.
------------------------------------------------------------

[Q3] 질문: PR #287 사용자 인증 모듈 리팩토링에 대한 리뷰 결정은 무엇이었나요?
[A3] 답변:
피드백 반영 후 재리뷰하며, 목요일까지 수정을 완료하는 것을 목표로 합니다.
------------------------------------------------------------

[Q4] 질문: ABC 테크놀로지 고객사의 월간 활성 사용자(MAU) 수치는 어떻게 되나요?
[A4] 답변:
ABC 테크놀로지 고객사의 월간 활성 사용자(MAU) 수치는 980명입니다.
------------------------------------------------------------

[Q5] 질문: 5월 14일 발생한 장애의 지속 시간과 영향 받은 사용자 수는?
[A5] 답변:
5월 14일 발생한 장애의 지속 시간은 2시간 24분이며, 영향 받은 사용자 수는 약 15,000명입니다.
------------------------------------------------------------

[Q6] 질문: 2024년 1분기 매출 실적과 목표 대비 달성률은 얼마인가요?
[A6] 답변:
2024년 1분기 매출 실적은 92억원이며, 목표 대비 달성률은 108%입니다.
------------------------------------------------------------

In [6]:
llm = init_chat_model('google_genai:gemini-3.1-flash-lite', api_key=GOOGLE_AI_API_KEY, temperature=0.1)
rag_chain = (
    {
        "context": compression_retriever | format_docs, 
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

# 4. 질문 리스트를 순회하며 테스트 실행
print("회의록 RAG 질의응답 테스트를 시작합니다.")
print("=" * 60)

for i, q in enumerate(test_questions, 1):
    print(f"\n[Q{i}] 질문: {q}")
    
    # RAG 검색 및 생성 실행
    response = rag_chain.invoke(q)
    
    # 가 Gemma 모델의 경우 가끔 내부 생각 구조(Thinking block)가 들어오거나 포맷이 정해져 있을 수 있으므로 그대로 출력합니다.
    print(f"[A{i}] 답변:")
    print(response)
    print("-" * 60)


회의록 RAG 질의응답 테스트를 시작합니다.

[Q1] 질문: 1월 킥오프 회의에서 결정된 최종 릴리즈 일정은 언제인가요?
[A1] 답변:
1월 킥오프 회의에서 결정된 최종 릴리즈 일정은 2024년 4월 30일입니다.
------------------------------------------------------------

[Q2] 질문: 스프린트 1에서 완료되지 못해 스프린트 2로 이월된 작업은 무엇인가요?
[A2] 답변:
스프린트 1에서 완료되지 못해 스프린트 2로 이월된 작업은 'API 인증 모듈'입니다.
------------------------------------------------------------

[Q3] 질문: PR #287 사용자 인증 모듈 리팩토링에 대한 리뷰 결정은 무엇이었나요?
[A3] 답변:
PR #287 사용자 인증 모듈 리팩토링에 대한 결정 사항은 '피드백 반영 후 재리뷰'이며, 목요일까지 수정을 완료하는 것을 목표로 합니다.
------------------------------------------------------------

[Q4] 질문: ABC 테크놀로지 고객사의 월간 활성 사용자(MAU) 수치는 어떻게 되나요?
[A4] 답변:
ABC 테크놀로지 고객사의 월간 활성 사용자(MAU) 수치는 980명입니다.
------------------------------------------------------------

[Q5] 질문: 5월 14일 발생한 장애의 지속 시간과 영향 받은 사용자 수는?
[A5] 답변:
5월 14일 발생한 장애의 지속 시간은 2시간 24분이며, 영향 받은 사용자 수는 약 15,000명입니다.
------------------------------------------------------------

[Q6] 질문: 2024년 1분기 매출 실적과 목표 대비 달성률은 얼마인가요?
[A6] 답변:
2024년 1분기 매출 실적은 92억원이며, 목표 대비 달성률은 108%입니다.
------

In [8]:
# ragas import 전에 임시 호환 코드
def patch_ragas_langchain_vertex_import() -> None:
    """
    ragas 0.4.x와 langchain-community 0.4.x 조합에서
    ragas import 시점에 없는 VertexAI 경로를 찾는 문제를 우회합니다.

    이 코드는 VertexAI를 사용하는 코드가 아닙니다.
    OpenAI만 쓰는 실습에서 ragas import가 실패하지 않게 하는 임시 호환 코드입니다.
    """
    import importlib.util
    import sys
    import types

    module_name = "langchain_community.chat_models.vertexai"

    if importlib.util.find_spec(module_name) is not None:
        return

    vertexai_module = types.ModuleType(module_name)

    class ChatVertexAI:
        def __init__(self, *args, **kwargs):
            raise ImportError(
                "ChatVertexAI is not installed. This notebook uses OpenAI only, "
                "so VertexAI should not be instantiated."
            )

    vertexai_module.ChatVertexAI = ChatVertexAI
    sys.modules[module_name] = vertexai_module

patch_ragas_langchain_vertex_import()

In [18]:
from datasets import Dataset
from ragas import evaluate
from langchain_core.runnables import RunnablePassthrough

# 1. 평가용 Ground Truth(실제 정답 기준) 정의
# (Ragas의 context_recall 등의 메트릭을 평가하기 위해 실제 정답이 필요합니다)
ground_truths = [
    ["최종 릴리즈 일정은 2024년 4월 30일입니다."],
    ["스프린트 1에서 이월된 작업은 API 인증 모듈입니다."],
    ["피드백 반영 후 재리뷰하며, 목요일까지 수정을 완료하는 것을 목표로 합니다."],
    ["ABC 테크놀로지 고객사의 월간 활성 사용자(MAU) 수치는 980명입니다."],
    ["5월 14일 발생한 장애의 지속 시간은 2시간 24분이며, 영향 받은 사용자 수는 약 15,000명입니다."],
    ["2024년 1분기 매출 실적은 92억원이며, 목표 대비 달성률은 108%입니다."],
    ["SmartWork Pro 4.0 고객 요청 기능 1순위는 '워크플로우 자동화'입니다."]
]

# 2. Ragas 입력용 데이터 수집
questions = test_questions
answers = []
contexts = []

print("[상태] Ragas 평가를 위한 데이터 수집을 시작합니다...")

for q in questions:
    # 2-1. 검색 단계 컨텍스트 추출
    retrieved_docs = compression_retriever.invoke(q)
    # Ragas는 각 질문당 검색된 문장들의 list[str] 형태를 입력받습니다.
    contexts.append([doc.page_content for doc in retrieved_docs])
    
    # 2-2. 최종 답변 생성
    response = rag_chain.invoke(q)
    answers.append(response)

# 3. Ragas Dataset 포맷 생성
data = {
    "question": questions,
    "answer": answers,
    "contexts": contexts,
    "ground_truth": [gt[0] for gt in ground_truths] # list[str] 형태로 플래싱
}
dataset = Dataset.from_dict(data)
print("[성공] Dataset 구축 완료")


[상태] Ragas 평가를 위한 데이터 수집을 시작합니다...


[성공] Dataset 구축 완료


In [ ]:
# 1. 소문자로 정의된 구형 인스턴스들을 임포트합니다.
# (이 객체들은 evaluate() 함수가 인식하는 Metric 클래스를 상속받고 있습니다)
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# 2. 메트릭 리스트를 구성합니다. (인스턴스이므로 소괄호 '()'를 붙이지 않습니다)
metrics = [
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
]

# 3. LangChain 객체를 다시 래핑해 줍니다.
ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

# 4. Ragas 평가 실행
print("[상태] Ragas 평가를 시작합니다...")
result = evaluate(
    dataset=dataset,
    metrics=metrics,
    llm=ragas_llm,
    embeddings=ragas_embeddings
)

# 5. 결과 시각화
print("\n================ [Ragas RAG 평가 결과] ================")
df_result = result.to_pandas()
df_result[[
    "user_input", 
    "response", 
    "faithfulness", 
    "answer_relevancy", 
    "context_precision", 
    "context_recall"
]]


In [22]:
df_result[[
    "user_input", 
    "response", 
    "faithfulness", 
    "answer_relevancy", 
    "context_precision", 
    "context_recall"
]]

,user_input,response,faithfulness,answer_relevancy,context_precision,context_recall
0,1월 킥오프 회의에서 결정된 최종 릴리즈 일정은 언제인가요?,1월 킥오프 회의에서 결정된 최종 릴리즈 일정은 2024년 4월 30일입니다.,1.0,1.000000,1.0,1.0
1,스프린트 1에서 완료되지 못해 스프린트 2로 이월된 작업은 무엇인가요?,스프린트 1에서 완료되지 못해 스프린트 2로 이월된 작업은 'API 인증 모듈'입니다.,1.0,1.000000,1.0,1.0
2,PR #287 사용자 인증 모듈 리팩토링에 대한 리뷰 결정은 무엇이었나요?,PR #287 사용자 인증 모듈 리팩토링에 대한 결정 사항은 '피드백 반영 후 재리...,1.0,0.850378,1.0,1.0
3,ABC 테크놀로지 고객사의 월간 활성 사용자(MAU) 수치는 어떻게 되나요?,ABC 테크놀로지 고객사의 월간 활성 사용자(MAU) 수치는 980명입니다.,0.0,0.963797,1.0,1.0
4,5월 14일 발생한 장애의 지속 시간과 영향 받은 사용자 수는?,"5월 14일 발생한 장애의 지속 시간은 2시간 24분이며, 영향 받은 사용자 수는 ...",1.0,0.993428,1.0,1.0
5,2024년 1분기 매출 실적과 목표 대비 달성률은 얼마인가요?,"2024년 1분기 매출 실적은 92억원이며, 목표 대비 달성률은 108%입니다.",1.0,0.991622,1.0,1.0
6,SmartWork Pro 4.0 고객 요청 기능 1순위는 무엇인가요?,SmartWork Pro 4.0 고객 요청 기능 1순위는 '워크플로우 자동화'입니다.,1.0,0.898848,1.0,1.0
